# Liu2024 Source MAT S-JEPA Self-Supervised Pretraining

S-JEPA-style masked latent prediction pretraining on Liu2024 source `.mat` EEG windows.

# 1. Setup

In [ ]:
import os
import re
import sys
import json
import math
import hashlib
import random
import builtins
import platform
import gc
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader, ConcatDataset

from scipy.io import loadmat
from scipy import signal

from braindecode.models import SignalJEPA

import mne

mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("Imports loaded successfully.")

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().resolve().parent.parent
print(f"\nWorking directory: {WORKING_DIR}")


# 2. Configuration

In [ ]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # Paths / identity
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-pretraining"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "liu2024_sjepa_self_supervised_pretraining_clean",
    "config_note": "Clean Liu2024 S-JEPA self-supervised pretraining. Uses unlabeled EEG windows only.",

    # Subject split. Keep downstream subjects untouched.
    "train_subject_ids": list(range(1, 31)),
    "val_subject_ids": list(range(31, 41)),
    "excluded_subject_ids": list(range(41, 51)),
    "paradigm_names": ["Liu2024_MI"],

    # Units and preprocessing. Keep this aligned with downstream Liu2024 runs.
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",
    "demean_mode": "none",
    "reference_mode": "average",
    "reference_timing": "before_resample_filter",
    "resample": True,
    "resample_sfreq": 128,
    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",

    # Pretraining windows.
    # fixed_crop uses mi_window_start_s. sliding uses pretrain_start_s/pretrain_stop_s and sampling_interval_s.
    "pretrain_window_samples": 537,
    "mi_window_start_s": 1.5,
    "pretrain_window_mode": "fixed_crop",  # fixed_crop, sliding
    "pretrain_start_s": 0.0,
    "pretrain_stop_s": 8.0,
    "sampling_interval_s": 1.0,

    # S-JEPA objective.
    "masking_strategy": "random_spatial_block_radius",
    "mask_diameter_percent": 60,
    "predictor_n_layers": 4,
    "predictor_nhead": 8,
    "predictor_dim_feedforward": 256,
    "ema_decay": 0.996,

    # Training.
    "train_subject_chunk_size": 30,
    "val_subject_chunk_size": 10,
    "chunk_shuffle": True,
    "window_preload": True,
    "batch_size": 16,
    "n_epochs": 300,
    "early_stopping_patience": 20,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,

    # Export.
    "export_with_chans": True,
    "export_without_chans": True,

    # Runtime.
    "device": "auto",
    "log_verbosity": "compact",
    "seed": 2026,
}


In [ ]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ

WINDOW_SAMPLES = int(CONFIG["pretrain_window_samples"])
CONFIG["pretrain_duration_s"] = WINDOW_SAMPLES / EFFECTIVE_SFREQ

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG.get("mi_window_start_s", 0.0)) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES
PRETRAIN_START_SAMPLE = int(round(float(CONFIG.get("pretrain_start_s", 0.0)) * EFFECTIVE_SFREQ))
PRETRAIN_STOP_SAMPLE = None if CONFIG.get("pretrain_stop_s", None) is None else int(round(float(CONFIG["pretrain_stop_s"]) * EFFECTIVE_SFREQ))

print("Effective Liu2024 S-JEPA pretraining settings:")
print(f"  Experiment:                  {CONFIG.get('experiment_name')}")
print(f"  Note:                        {CONFIG.get('config_note')}")
print(f"  Channels:                    {len(EEG_CHANNEL_NAMES)}")
print(f"  Source sfreq:                {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:             {EFFECTIVE_SFREQ} Hz")
print(f"  Window samples:              {WINDOW_SAMPLES}")
print(f"  Window duration:             {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Window mode:                 {CONFIG.get('pretrain_window_mode')}")
print(f"  Fixed crop start:            {CONFIG.get('mi_window_start_s')} s")
print(f"  Sliding range:               {CONFIG.get('pretrain_start_s')}–{CONFIG.get('pretrain_stop_s')} s")
print(f"  Sampling interval:           {CONFIG.get('sampling_interval_s')} s")
print(f"  Train subjects:              {CONFIG['train_subject_ids']}")
print(f"  Validation subjects:         {CONFIG['val_subject_ids']}")
print(f"  Excluded/downstream only:    {CONFIG.get('excluded_subject_ids')}")
print(f"  Preprocessing:               {CONFIG['reference_mode']} reference, "
      f"{CONFIG['filter_low']}–{CONFIG['filter_high']} Hz {CONFIG['filter_method']}, "
      f"resample={CONFIG['resample']}->{CONFIG.get('resample_sfreq')} Hz, demean={CONFIG['demean_mode']}")
print(f"  Training:                    batch={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}, "
      f"epochs={CONFIG['n_epochs']}, patience={CONFIG['early_stopping_patience']}")


In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


In [ ]:
def resolve_device():
    requested = str(CONFIG.get("device", "auto")).lower()
    if requested == "cpu":
        return torch.device("cpu")
    if requested == "cuda":
        return torch.device("cuda")
    if requested == "mps":
        return torch.device("mps")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

def assert_finite_tensor(name, tensor):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"Non-finite tensor detected: {name}")

LOG_VERBOSITY = str(CONFIG.get("log_verbosity", "compact")).lower()

def log_debug(message):
    if LOG_VERBOSITY == "debug":
        print(message)

set_seed(int(CONFIG.get("seed", 42)))

# 3. Load and Prepare Liu2024 Windows

In [ ]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


In [ ]:

def force_signal_jepa_safe_positions(chs_info):
    """Force finite, non-degenerate channel coordinates for SignalJEPA.

    Braindecode SignalJEPA normalizes each coordinate axis as
    (x - min) / (max - min). If any axis has zero range, positional encoding
    becomes NaN. This function overwrites loc[:3] directly and guarantees
    non-zero range on x/y/z.
    """
    montage_alias = {
        "T3": "T7",
        "T4": "T8",
        "T5": "P7",
        "T6": "P8",
    }

    montage = mne.channels.make_standard_montage("standard_1020")
    ch_pos = montage.get_positions()["ch_pos"]

    xyz_rows = []
    missing = []

    for ch in chs_info:
        name = ch["ch_name"]
        lookup_name = montage_alias.get(name, name)

        if lookup_name in ch_pos:
            xyz = np.asarray(ch_pos[lookup_name], dtype=np.float64)
        else:
            missing.append(name)
            xyz = np.zeros(3, dtype=np.float64)

        xyz_rows.append(xyz)

    if missing:
        raise RuntimeError(f"Missing montage coordinates for channels: {missing}")

    xyz_rows = np.asarray(xyz_rows, dtype=np.float64)
    if not np.isfinite(xyz_rows).all():
        raise RuntimeError("Non-finite coordinates before SignalJEPA position safety fix.")

    # Guarantee every coordinate axis has non-zero range. This avoids the
    # divide-by-zero in braindecode.models.signal_jepa.py positional encoding.
    n_channels = xyz_rows.shape[0]
    theta = np.linspace(0.0, 2.0 * np.pi, n_channels, endpoint=False)

    for axis in range(3):
        axis_range = float(xyz_rows[:, axis].max() - xyz_rows[:, axis].min())
        if axis_range <= 1e-8:
            if axis == 0:
                xyz_rows[:, axis] = np.cos(theta)
            elif axis == 1:
                xyz_rows[:, axis] = np.sin(theta)
            else:
                xyz_rows[:, axis] = np.linspace(-0.5, 0.5, n_channels)

    for ch, xyz in zip(chs_info, xyz_rows):
        loc = np.zeros(12, dtype=np.float64)
        loc[:3] = xyz
        ch["loc"][:] = loc

    xyz_range = xyz_rows.max(axis=0) - xyz_rows.min(axis=0)

    print("SignalJEPA-safe channel coordinate range:")
    print(f"  x range: {xyz_range[0]:.6f}")
    print(f"  y range: {xyz_range[1]:.6f}")
    print(f"  z range: {xyz_range[2]:.6f}")

    if not np.isfinite(xyz_rows).all() or np.any(xyz_range <= 1e-8):
        raise RuntimeError(f"SignalJEPA-safe position fix failed: xyz_range={xyz_range.tolist()}")

    return chs_info


def make_liu_info(sfreq):
    """Build MNE info for Liu2024 channels with finite channel locations.

    Liu2024 uses older temporal channel names (T3/T4/T5/T6). MNE standard_1020
    uses the modern equivalents (T7/T8/P7/P8), so the names are temporarily
    mapped for montage lookup and then renamed back.

    Important: MNE channel loc arrays often contain NaNs outside loc[:3].
    SignalJEPA's positional encoder can propagate any NaN in chs_info, so the
    entire loc vector is sanitized, not only the xyz coordinates.
    """
    montage_alias = {
        "T3": "T7",
        "T4": "T8",
        "T5": "P7",
        "T6": "P8",
    }

    montage_ch_names = [
        montage_alias.get(name, name)
        for name in EEG_CHANNEL_NAMES
    ]

    info = mne.create_info(
        ch_names=montage_ch_names,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(montage_ch_names),  # type: ignore
    )

    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="warn")

    rename_back = {
        modern_name: old_name
        for old_name, modern_name in montage_alias.items()
        if old_name in EEG_CHANNEL_NAMES and modern_name in info.ch_names
    }
    if rename_back:
        info.rename_channels(rename_back)

    for ch in info["chs"]:
        loc = np.asarray(ch["loc"], dtype=np.float64)
        if loc.shape[0] == 0:
            ch["loc"] = np.zeros(12, dtype=np.float64)
            print(f"WARNING: empty location for channel {ch['ch_name']}; replacing with zeros.")
            continue

        if not np.isfinite(loc).all():
            bad_count = int((~np.isfinite(loc)).sum())
            print(
                f"WARNING: replacing {bad_count} missing/non-finite location values "
                f"for channel {ch['ch_name']} with zeros."
            )
            ch["loc"][:] = np.nan_to_num(loc, nan=0.0, posinf=0.0, neginf=0.0)

    return info



def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "pretraining_window",
        "mode": config.get("pretrain_window_mode"),
        "fixed_start_s": config.get("mi_window_start_s"),
        "window_samples": config.get("pretrain_window_samples"),
        "sliding_start_s": config.get("pretrain_start_s"),
        "sliding_stop_s": config.get("pretrain_stop_s"),
        "sampling_interval_s": config.get("sampling_interval_s"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    window_samples = int(config.get("pretrain_window_samples") or config.get("target_window_samples") or round(float(config["target_window_s"]) * effective_sfreq))
    window_mode = str(config.get("pretrain_window_mode", "fixed_crop")).lower()
    interval_samples = int(round(float(config.get("sampling_interval_s", 1.0)) * effective_sfreq))

    if window_mode == "fixed_crop":
        start_sample = int(round(float(config.get("mi_window_start_s", 0.0)) * effective_sfreq))
        starts = [start_sample]
    elif window_mode == "sliding":
        start_sample = int(round(float(config.get("pretrain_start_s", 0.0)) * effective_sfreq))
        stop_limit = config.get("pretrain_stop_s", None)
        if stop_limit is None:
            stop_limit_sample = X_rs.shape[-1]
        else:
            stop_limit_sample = int(round(float(stop_limit) * effective_sfreq))
        last_start = stop_limit_sample - window_samples
        if last_start < start_sample:
            raise ValueError(
                f"Subject {subject_id}: sliding range [{start_sample}:{stop_limit_sample}] "
                f"is shorter than window_samples={window_samples}"
            )
        starts = list(range(start_sample, last_start + 1, interval_samples))
    elif window_mode == "random_crop":
        rng = np.random.default_rng(int(config.get("seed", 0)) + int(subject_id))
        start_sample = int(round(float(config.get("pretrain_start_s", 0.0)) * effective_sfreq))
        stop_limit = config.get("pretrain_stop_s", None)
        if stop_limit is None:
            stop_limit_sample = X_rs.shape[-1]
        else:
            stop_limit_sample = int(round(float(stop_limit) * effective_sfreq))
        last_start = stop_limit_sample - window_samples
        if last_start < start_sample:
            raise ValueError(
                f"Subject {subject_id}: random-crop range [{start_sample}:{stop_limit_sample}] "
                f"is shorter than window_samples={window_samples}"
            )
        starts = [int(rng.integers(start_sample, last_start + 1))]
    else:
        raise ValueError(f"Unsupported pretrain_window_mode={config.get('pretrain_window_mode')}")

    window_list = []
    for start_sample in starts:
        stop_sample = start_sample + window_samples
        if stop_sample > X_rs.shape[-1]:
            raise ValueError(
                f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
                f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
            )
        window_list.append(X_rs[:, :, start_sample:stop_sample])

    X_win = np.concatenate(window_list, axis=0)
    y = np.tile(labels_to_zero_based(labels), len(window_list))
    runtime_steps.append(
        f"pretraining windows mode={window_mode} starts={starts} "
        f"window_samples={window_samples} interval_samples={interval_samples}"
    )

    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


In [ ]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


class TrainAugmentedDataset(Dataset):
    """Lightweight training-time augmentation for tiny subject-level MI data.

    The augmentation is intentionally simple and shape-safe:
      - Gaussian noise proportional to per-window std
      - small random temporal roll
      - random channel dropout
    """

    def __init__(
        self,
        dataset,
        noise_fraction: float = 0.0,
        time_shift_samples: int = 0,
        channel_dropout_prob: float = 0.0,
    ):
        self.dataset = dataset
        self.noise_fraction = float(noise_fraction or 0.0)
        self.time_shift_samples = int(time_shift_samples or 0)
        self.channel_dropout_prob = float(channel_dropout_prob or 0.0)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Augmentation expects C x T, got shape={x.shape}.")

        if self.time_shift_samples > 0:
            shift = np.random.randint(-self.time_shift_samples, self.time_shift_samples + 1)
            if shift != 0:
                x = np.roll(x, shift=shift, axis=-1)

        if self.channel_dropout_prob > 0:
            mask = np.random.rand(x.shape[0]) >= self.channel_dropout_prob
            if not mask.any():
                mask[np.random.randint(0, x.shape[0])] = True
            x = x.copy()
            x[~mask, :] = 0.0

        if self.noise_fraction > 0:
            sigma = self.noise_fraction * float(np.std(x) + 1e-8)
            noise = np.random.randn(*x.shape).astype(np.float32) * sigma
            x = x + noise

        return x.astype(np.float32), int(y)


# Backward compatibility for older code.
NoisyDataset = TrainAugmentedDataset


In [ ]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
PRETRAIN_SUBJECT_SET = set(int(s) for s in CONFIG["train_subject_ids"]) | set(int(s) for s in CONFIG["val_subject_ids"])

for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if sid not in PRETRAIN_SUBJECT_SET:
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CHS_INFO = force_signal_jepa_safe_positions(CHS_INFO)

bad_locs = []
xyz_rows = []
for ch in CHS_INFO:
    loc = np.asarray(ch["loc"], dtype=np.float64)
    if not np.isfinite(loc).all():
        bad_locs.append(ch["ch_name"])
    xyz_rows.append(loc[:3])

xyz_rows = np.asarray(xyz_rows, dtype=np.float64)
xyz_range = xyz_rows.max(axis=0) - xyz_rows.min(axis=0)

print(f"Channels with non-finite location vectors: {bad_locs}")
print(f"CHS_INFO xyz coordinate range: {xyz_range.tolist()}")

if bad_locs:
    raise RuntimeError(f"Non-finite channel locations remain after SignalJEPA position fix: {bad_locs}")
if np.any(xyz_range <= 1e-8):
    raise RuntimeError(f"Degenerate channel coordinates remain after SignalJEPA position fix: {xyz_range.tolist()}")

CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "pretrain_start_sample": int(PRETRAIN_START_SAMPLE),
        "pretrain_stop_sample": None if PRETRAIN_STOP_SAMPLE is None else int(PRETRAIN_STOP_SAMPLE),
        "pretrain_window_mode": CONFIG.get("pretrain_window_mode"),
        "pretrain_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


# 4. Pretraining Data Streaming

In [ ]:
TRAIN_SUBJECTS = [int(s) for s in CONFIG["train_subject_ids"]]
VAL_SUBJECTS = [int(s) for s in CONFIG["val_subject_ids"]]
DOWNSTREAM_ONLY_SUBJECTS = [int(s) for s in CONFIG.get("excluded_subject_ids", [])]

def iter_subject_chunks(subject_ids, chunk_size, shuffle=False, seed=None):
    if chunk_size <= 0:
        raise ValueError(f"chunk_size must be > 0, got {chunk_size}")
    ordered_subjects = [int(subject_id) for subject_id in subject_ids]
    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(ordered_subjects)
    for start_idx in range(0, len(ordered_subjects), chunk_size):
        chunk = ordered_subjects[start_idx : start_idx + chunk_size]
        if chunk:
            yield chunk

def compute_pretraining_window_params(sfreq, pretrain_duration_s=None, sampling_interval_s=1.0, pretrain_window_samples=None):
    """Compute pretraining window parameters.

    Use pretrain_window_samples when supplied. This avoids small rounding
    differences and keeps S-JEPA token/window compatibility explicit.
    """
    sfreq = float(sfreq)
    sampling_interval_s = float(sampling_interval_s)

    if pretrain_window_samples is not None:
        window_size_samples = int(pretrain_window_samples)
        pretrain_duration_s = window_size_samples / sfreq
        CONFIG["pretrain_duration_s"] = float(pretrain_duration_s)
    else:
        pretrain_duration_s = float(pretrain_duration_s)
        window_size_samples = round(pretrain_duration_s * sfreq)

    sampling_interval_samples = round(sampling_interval_s * sfreq)

    if window_size_samples <= 0:
        raise ValueError(f"window_size_samples must be > 0, got {window_size_samples}")
    if sampling_interval_samples <= 0:
        raise ValueError(f"sampling_interval_samples must be > 0, got {sampling_interval_samples}")

    print("Pretraining window parameters:")
    print(f"  sfreq:                      {sfreq} Hz")
    print(f"  pretrain_duration_s:        {pretrain_duration_s} s")
    print(f"  sampling_interval_s:        {sampling_interval_s} s")
    print(f"  window_size_samples:        {window_size_samples}")
    print(f"  sampling_interval_samples:  {sampling_interval_samples}")

    return window_size_samples, sampling_interval_samples


WINDOW_SIZE_SAMPLES, SAMPLING_INTERVAL_SAMPLES = compute_pretraining_window_params(
    sfreq=CONFIG["sfreq"],
    pretrain_duration_s=CONFIG.get("pretrain_duration_s"),
    sampling_interval_s=CONFIG["sampling_interval_s"],
    pretrain_window_samples=CONFIG.get("pretrain_window_samples"),
)


if WINDOW_SIZE_SAMPLES != WINDOW_SAMPLES:
    raise RuntimeError(
        f"CONFIG mismatch: pretraining window samples={WINDOW_SIZE_SAMPLES}, "
        f"Liu window samples={WINDOW_SAMPLES}."
    )

def prepare_pretraining_chunk_dataset(
    paradigm_names,
    subject_ids,
    split_name,
    preprocessors,
    window_size_samples,
    sampling_interval_samples,
    preload,
):
    # Windows were created once in preprocess_subject_configurable above.
    # window_size_samples/sampling_interval_samples are kept in the signature only
    # to preserve the training-loop call style.

    datasets = []
    missing_subjects = []
    for subject_id in subject_ids:
        key = str(int(subject_id))
        if key not in SUBJECT_WINDOWS:
            missing_subjects.append(subject_id)
        else:
            datasets.append(SUBJECT_WINDOWS[key])

    if missing_subjects:
        raise RuntimeError(f"{split_name} missing subjects from SUBJECT_WINDOWS: {missing_subjects}")
    if not datasets:
        raise RuntimeError(f"{split_name} chunk has no datasets.")

    chunk_windows = ConcatDataset(datasets)
    chunk_raw_dataset = SimpleNamespace(datasets=datasets)
    recordings_by_paradigm = {"Liu2024_MI": len(datasets)}
    log_debug(
        f"[debug] loaded {split_name} Liu chunk: subjects={subject_ids} "
        f"windows={len(chunk_windows)} recordings={len(datasets)}"
    )
    return chunk_windows, chunk_raw_dataset, recordings_by_paradigm

def make_chunk_dataloader(windows_dataset, batch_size, training):
    return DataLoader(
        windows_dataset,
        batch_size=batch_size,
        shuffle=bool(training),
        num_workers=0,
        drop_last=bool(training),
    )

_reference_subject_chunk = next(
    iter_subject_chunks(
        TRAIN_SUBJECTS,
        chunk_size=CONFIG["train_subject_chunk_size"],
        shuffle=False,
    )
)

REFERENCE_WINDOWS_DATASET, REFERENCE_RAW_DATASET, _reference_recordings = prepare_pretraining_chunk_dataset(
    paradigm_names=CONFIG["paradigm_names"],
    subject_ids=_reference_subject_chunk,
    split_name="reference",
    preprocessors=None,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
    preload=CONFIG["window_preload"],
)

CH_POSITIONS_np = []
CH_VALID_MASK_np = []
for ch in CHS_INFO:
    loc = ch.get("loc", None)
    if loc is None or len(loc) < 3:
        xyz = np.zeros(3, dtype=np.float32)
    else:
        xyz = np.asarray(loc[:3], dtype=np.float32)
        if not np.isfinite(xyz).all():
            xyz = np.zeros(3, dtype=np.float32)
    CH_POSITIONS_np.append(xyz)
    CH_VALID_MASK_np.append(bool(np.linalg.norm(xyz) > 1e-8))

CH_POSITIONS = torch.tensor(np.stack(CH_POSITIONS_np), dtype=torch.float32)
CH_VALID_MASK = torch.tensor(np.asarray(CH_VALID_MASK_np), dtype=torch.bool)
N_CHANNELS = len(CH_NAMES)

print("Liu2024 S-JEPA pretraining split:")
print(f"  Train subjects:                {TRAIN_SUBJECTS}")
print(f"  Validation subjects:           {VAL_SUBJECTS}")
print(f"  Excluded subjects:             {DOWNSTREAM_ONLY_SUBJECTS}")
print(f"  Reference subjects:            {_reference_subject_chunk}")
print(f"  Reference windows:             {len(REFERENCE_WINDOWS_DATASET)}")
print(f"  EEG channels:                  {N_CHANNELS}")
print(f"  Valid channel positions:       {int(CH_VALID_MASK.sum().item())}/{len(CH_VALID_MASK)}")

In [ ]:
_PREPROCESSORS = None
print("Liu2024 windows are already preprocessed by the source MAT pipeline above.")

# 5. Model

In [ ]:

def reset_signal_jepa_spatial_embedding_if_needed(model, chs_info, name="model"):
    """Repair NaN/inf spatial positional embeddings after SignalJEPA construction.

    Braindecode initializes the channel embedding from channel locations during
    SignalJEPA construction. If the constructor receives degenerate locations,
    the embedding can become NaN before we ever call forward(). This function
    directly reinitializes the spatial embedding table with a finite sinusoidal
    code derived from CHS_INFO.
    """
    if not hasattr(model, "pos_encoder") or model.pos_encoder is None:
        return False
    if not hasattr(model.pos_encoder, "pos_encoder_spat"):
        return False

    emb = model.pos_encoder.pos_encoder_spat
    with torch.no_grad():
        if torch.isfinite(emb.weight).all():
            print(f"{name} spatial positional embedding already finite.")
            return False

        n_embeddings, spat_dim = emb.weight.shape
        xyz_rows = []
        for ch in chs_info[:n_embeddings]:
            loc = np.asarray(ch["loc"], dtype=np.float64)
            xyz_rows.append(loc[:3])

        xyz = np.asarray(xyz_rows, dtype=np.float64)
        if xyz.shape != (n_embeddings, 3):
            raise RuntimeError(
                f"Unexpected xyz shape for positional reinit: {xyz.shape}, "
                f"expected {(n_embeddings, 3)}"
            )

        xyz = np.nan_to_num(xyz, nan=0.0, posinf=0.0, neginf=0.0)
        global_abs = float(np.max(np.abs(xyz)))
        if not np.isfinite(global_abs) or global_abs <= 1e-8:
            global_abs = 1.0

        if spat_dim % 3 != 0:
            raise RuntimeError(f"Expected spat_dim divisible by 3, got {spat_dim}")
        dim_per_coord = spat_dim // 3
        if dim_per_coord % 2 != 0:
            raise RuntimeError(f"Expected dim_per_coord even, got {dim_per_coord}")

        code = torch.empty(
            (n_embeddings, spat_dim),
            dtype=emb.weight.dtype,
            device=emb.weight.device,
        )

        div_term = torch.exp(
            (1.0 - torch.arange(0, dim_per_coord, 2, device=emb.weight.device, dtype=emb.weight.dtype) / dim_per_coord)
            * (2.0 * np.pi)
        )

        xyz_tensor = torch.as_tensor(
            xyz,
            dtype=emb.weight.dtype,
            device=emb.weight.device,
        )

        # Use a global safe range [-global_abs, global_abs] so the denominator
        # is never zero even if one coordinate axis is constant.
        for coord_idx in range(3):
            start = coord_idx * dim_per_coord
            stop = start + dim_per_coord
            xx = (xyz_tensor[:, coord_idx] + global_abs) / (2.0 * global_abs)
            code[:, start:stop:2] = torch.sin(xx[:, None] * div_term[None, :])
            code[:, start + 1:stop:2] = torch.cos(xx[:, None] * div_term[None, :])

        emb.weight.copy_(code)

        if not torch.isfinite(emb.weight).all():
            raise RuntimeError(f"{name} spatial positional embedding is still non-finite after repair.")

        print(f"{name} spatial positional embedding repaired and is finite.")
        return True


def compute_n_tok_per_channel(conv_layers_spec, n_times):
    """Compute the number of feature-encoder output tokens per EEG channel."""
    n_times_out = n_times
    for _, kernel_size, stride in conv_layers_spec:
        n_times_out = (n_times_out - kernel_size) // stride + 1
    return n_times_out


def build_signal_jepa_backbone(chs_info, sfreq, input_window_seconds):
    """Instantiate a fresh SignalJEPA backbone."""
    return SignalJEPA(
        sfreq=sfreq,
        input_window_seconds=input_window_seconds,
        chs_info=chs_info,
    )

In [ ]:
_DEFAULT_CONV_LAYER_SPEC = (
    (8, 32, 8),
    (16, 2, 2),
    (32, 2, 2),
    (64, 2, 2),
    (64, 2, 2),
)

N_TOK_PER_CHANNEL = compute_n_tok_per_channel(_DEFAULT_CONV_LAYER_SPEC, WINDOW_SIZE_SAMPLES)
EMB_DIM = _DEFAULT_CONV_LAYER_SPEC[-1][0]
TOTAL_TOKENS = N_CHANNELS * N_TOK_PER_CHANNEL

if N_TOK_PER_CHANNEL < 2:
    raise RuntimeError(
        f"SignalJEPA positional encoding requires at least 2 tokens per channel, "
        f"but WINDOW_SIZE_SAMPLES={WINDOW_SIZE_SAMPLES} gives "
        f"N_TOK_PER_CHANNEL={N_TOK_PER_CHANNEL}. Increase pretrain_window_samples. "
        f"With the current feature encoder at 128 Hz, use at least 280 samples."
    )

print("Feature encoder token geometry:")
print(f"  WINDOW_SIZE_SAMPLES:  {WINDOW_SIZE_SAMPLES}")
print(f"  N_TOK_PER_CHANNEL:    {N_TOK_PER_CHANNEL}")
print(f"  EMB_DIM:              {EMB_DIM}")
print(f"  TOTAL_TOKENS:         {TOTAL_TOKENS}")

In [ ]:
INPUT_WINDOW_SECONDS = WINDOW_SIZE_SAMPLES / CONFIG["sfreq"]

STUDENT = build_signal_jepa_backbone(
    chs_info=CHS_INFO,
    sfreq=CONFIG["sfreq"],
    input_window_seconds=INPUT_WINDOW_SECONDS,
).to(DEVICE)
reset_signal_jepa_spatial_embedding_if_needed(STUDENT, CHS_INFO, name="student")
STUDENT.train()

TEACHER = deepcopy(STUDENT).to(DEVICE)
reset_signal_jepa_spatial_embedding_if_needed(TEACHER, CHS_INFO, name="teacher")
TEACHER.eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad = False

student_param_count = sum(parameter.numel() for parameter in STUDENT.parameters())
teacher_trainable_count = sum(
    parameter.numel() for parameter in TEACHER.parameters() if parameter.requires_grad
)

print("Student / teacher backbone summary:")
print(f"  Student parameters:           {student_param_count:,}")
print(f"  Feature encoder parameters:   {sum(p.numel() for p in STUDENT.feature_encoder.parameters()):,}") # type: ignore
print(f"  Positional encoder params:    {sum(p.numel() for p in STUDENT.pos_encoder.parameters()):,}") # type: ignore
print(f"  Transformer parameters:       {sum(p.numel() for p in STUDENT.transformer.parameters()):,}") # type: ignore
print(f"  Teacher trainable parameters: {teacher_trainable_count}")

# Probe one forward path early so positional-encoding issues fail before training.
_probe_x = torch.from_numpy(REFERENCE_WINDOWS_DATASET[0][0]).float().unsqueeze(0).to(DEVICE)
with torch.no_grad():
    _probe_local = STUDENT.feature_encoder(_probe_x)
    print(f"Actual feature encoder output shape: {tuple(_probe_local.shape)}")
    _probe_pos = STUDENT.pos_encoder(_probe_local)
    print(f"probe_pos nonfinite count: {int((~torch.isfinite(_probe_pos)).sum().item())}")
    _probe_full = _probe_local + _probe_pos
    _probe_context = TEACHER.transformer.encoder(_probe_full)

assert_finite_tensor("probe_local", _probe_local)
assert_finite_tensor("probe_pos", _probe_pos)
assert_finite_tensor("probe_full", _probe_full)
assert_finite_tensor("probe_context", _probe_context)
print("Backbone finite probe: OK")

In [ ]:
class RandomSpatialBlockMaskSampler():
    """Radius-based random spatial block masking in EEG channel space."""

    def __init__(
        self,
        ch_positions: torch.Tensor,
        ch_names,
        mask_diameter_percent: float,
        n_tok_per_channel: int,
        valid_position_mask: torch.Tensor | None = None,
    ):
        self.ch_positions = ch_positions.clone().float().cpu()
        self.ch_names = list(ch_names)
        self.n_channels = int(self.ch_positions.shape[0])
        self.n_tok_per_channel = int(n_tok_per_channel)
        self.mask_diameter_percent = float(mask_diameter_percent)

        if len(self.ch_names) != self.n_channels:
            raise RuntimeError("ch_names length must match ch_positions first dimension.")
        if self.ch_positions.ndim != 2 or self.ch_positions.shape[1] != 3:
            raise RuntimeError(f"ch_positions must be (C, 3), got {tuple(self.ch_positions.shape)}")

        if valid_position_mask is None:
            inferred_valid = torch.isfinite(self.ch_positions).all(dim=-1) & (self.ch_positions.norm(dim=-1) > 1e-8)
        else:
            inferred_valid = valid_position_mask.clone().bool().cpu()

        if inferred_valid.shape != (self.n_channels,):
            raise RuntimeError("valid_position_mask must have shape (C,).")

        self.valid_position_mask = inferred_valid
        self.valid_indices = torch.where(self.valid_position_mask)[0]
        self.n_valid_channels = int(self.valid_indices.numel())
        if self.n_valid_channels < 2:
            raise RuntimeError(
                "Spatial masking requires at least two EEG channels with valid coordinates."
            )

        valid_positions = self.ch_positions[self.valid_indices]
        diffs = valid_positions.unsqueeze(1) - valid_positions.unsqueeze(0)
        self.valid_dist_matrix = diffs.norm(dim=-1)
        self.head_diameter = float(self.valid_dist_matrix.max().item())
        self.mask_diameter = self.head_diameter * (self.mask_diameter_percent / 100.0)
        self.mask_radius = self.mask_diameter / 2.0

        if self.head_diameter <= 0.0 or not np.isfinite(self.head_diameter):
            raise RuntimeError("Head diameter is invalid. EEG geometry is not trustworthy.")

    @staticmethod
    def _expand_channel_mask(mask_ch: torch.Tensor, n_tok_per_channel: int) -> torch.Tensor:
        return mask_ch.unsqueeze(-1).expand(-1, n_tok_per_channel).reshape(-1)

    def _compute_mask_from_center(self, center_global_idx: int, device):
        center_xyz = self.ch_positions[center_global_idx]
        distances = (self.ch_positions - center_xyz.unsqueeze(0)).norm(dim=-1)

        mask_ch = torch.zeros(self.n_channels, dtype=torch.bool)
        valid_distances = distances[self.valid_position_mask]
        mask_ch[self.valid_position_mask] = valid_distances <= (self.mask_radius + 1e-8)

        if not bool(mask_ch[center_global_idx]):
            mask_ch[center_global_idx] = True

        n_masked_valid = int(mask_ch[self.valid_position_mask].sum().item())
        if n_masked_valid == self.n_valid_channels:
            farthest_valid_rel = int(torch.argmax(valid_distances).item())
            farthest_valid_idx = int(self.valid_indices[farthest_valid_rel].item())
            if farthest_valid_idx != center_global_idx:
                mask_ch[farthest_valid_idx] = False

        mask_tok = self._expand_channel_mask(mask_ch, self.n_tok_per_channel)
        self.validate_single_mask(center_global_idx, mask_ch, mask_tok)

        return center_global_idx, mask_ch.to(device), mask_tok.to(device), distances.to(device)

    def validate_single_mask(self, center_global_idx: int, mask_ch: torch.Tensor, mask_tok: torch.Tensor):
        """Strong sanity checks for one sampled spatial mask."""
        if mask_ch.shape != (self.n_channels,):
            raise RuntimeError(f"mask_ch shape mismatch: {tuple(mask_ch.shape)}")
        if mask_tok.shape != (self.n_channels * self.n_tok_per_channel,):
            raise RuntimeError(f"mask_tok shape mismatch: {tuple(mask_tok.shape)}")

        center_xyz = self.ch_positions[center_global_idx]
        distances = (self.ch_positions - center_xyz.unsqueeze(0)).norm(dim=-1)

        if not bool(mask_ch[center_global_idx]):
            raise RuntimeError("Mask sanity failed: center channel is not masked.")

        valid_masked = mask_ch & self.valid_position_mask
        valid_unmasked = (~mask_ch) & self.valid_position_mask

        if bool(valid_masked.any()):
            if bool((distances[valid_masked] > (self.mask_radius + 1e-8)).any()):
                raise RuntimeError("Mask sanity failed: masked valid channel found outside radius.")
        if bool(valid_unmasked.any()):
            if bool((distances[valid_unmasked] <= (self.mask_radius + 1e-8)).any()):
                raise RuntimeError("Mask sanity failed: unmasked valid channel found inside radius.")

        expected_masked_tokens = int(mask_ch.sum().item()) * self.n_tok_per_channel
        observed_masked_tokens = int(mask_tok.sum().item())
        if observed_masked_tokens != expected_masked_tokens:
            raise RuntimeError(
                "Mask sanity failed: masked token count does not match "
                "masked_channel_count * n_tok_per_channel."
            )

    def sample(self, batch_size: int, device):
        center_indices = []
        center_channel_names = []
        mask_ch_list = []
        mask_tok_list = []
        masked_channel_counts = []
        masked_token_counts = []

        for _ in range(batch_size):
            center_valid_rel = int(torch.randint(0, self.n_valid_channels, (1,)).item())
            center_global_idx = int(self.valid_indices[center_valid_rel].item())
            center_global_idx, mask_ch, mask_tok, _ = self._compute_mask_from_center(center_global_idx, device)

            center_indices.append(center_global_idx)
            center_channel_names.append(self.ch_names[center_global_idx])
            mask_ch_list.append(mask_ch)
            mask_tok_list.append(mask_tok)
            masked_channel_counts.append(int(mask_ch.sum().item()))
            masked_token_counts.append(int(mask_tok.sum().item()))

        mask_ch = torch.stack(mask_ch_list, dim=0)
        mask_tok = torch.stack(mask_tok_list, dim=0)
        mask_info = {
            "center_indices": center_indices,
            "center_channel_names": center_channel_names,
            "masked_channel_counts": masked_channel_counts,
            "masked_token_counts": masked_token_counts,
        }
        return mask_ch, mask_tok, mask_info

    def sample_with_center(self, center_channel_name=None, center_idx=None, device="cpu"):
        """Sample one mask using either a named center or a center index."""
        if center_channel_name is not None:
            if center_channel_name not in self.ch_names:
                raise ValueError(f"Unknown center channel name: {center_channel_name}")
            center_idx = self.ch_names.index(center_channel_name)
        if center_idx is None:
            center_rel = int(torch.randint(0, self.n_valid_channels, (1,)).item())
            center_idx = int(self.valid_indices[center_rel].item())

        center_idx, mask_ch, mask_tok, distances = self._compute_mask_from_center(int(center_idx), device)
        mask_info = {
            "center_indices": [center_idx],
            "center_channel_names": [self.ch_names[center_idx]],
            "masked_channel_counts": [int(mask_ch.sum().item())],
            "masked_token_counts": [int(mask_tok.sum().item())],
        }
        return mask_ch.unsqueeze(0), mask_tok.unsqueeze(0), mask_info, distances

In [ ]:
def check_mask(mask_sampler: RandomSpatialBlockMaskSampler, center_channel_name: str):
    """
    Validate one mask for one chosen center electrode and print simple stats.
    """
    mask_ch, mask_tok, mask_info, distances = mask_sampler.sample_with_center(
        center_channel_name=center_channel_name,
        device="cpu",
    )

    mask_ch_single = mask_ch[0].cpu()
    mask_tok_single = mask_tok[0].cpu()
    distances_np = distances.cpu().numpy()

    center_idx = int(mask_info["center_indices"][0])
    center_name = mask_info["center_channel_names"][0]

    # hard validation using your sampler's own checks
    mask_sampler.validate_single_mask(
        center_global_idx=center_idx,
        mask_ch=mask_ch_single,
        mask_tok=mask_tok_single,
    )

    masked_indices = np.where(mask_ch_single.numpy())[0]
    masked_names = [mask_sampler.ch_names[i] for i in masked_indices]

    print(f"Mask check: {center_name}")
    print(f"  radius: {mask_sampler.mask_radius:.6f}")
    print(f"  masked channels: {len(masked_names)}")
    print(f"  masked tokens: {int(mask_tok_single.sum().item())}")
    print(f"  masked names: {masked_names}")

    return {
        "center_channel": center_name,
        "center_idx": center_idx,
        "radius": float(mask_sampler.mask_radius),
        "masked_channel_count": int(mask_ch_single.sum().item()),
        "masked_token_count": int(mask_tok_single.sum().item()),
        "masked_channels": masked_names,
        "distances": distances_np,
        "mask_ch": mask_ch_single.numpy(),
        "mask_tok": mask_tok_single.numpy(),
    }

In [ ]:
MASK_SAMPLER = RandomSpatialBlockMaskSampler(
    ch_positions=CH_POSITIONS,
    ch_names=CH_NAMES,
    mask_diameter_percent=CONFIG["mask_diameter_percent"],
    n_tok_per_channel=N_TOK_PER_CHANNEL,
    valid_position_mask=CH_VALID_MASK,
)

print("Radius-based random spatial block mask sampler:")
print(f"  n_channels:               {MASK_SAMPLER.n_channels}")
print(f"  n_valid_channels:         {MASK_SAMPLER.n_valid_channels}")
print(f"  head_diameter (valid):    {MASK_SAMPLER.head_diameter:.6f}")
print(f"  mask_diameter_percent:    {MASK_SAMPLER.mask_diameter_percent:.0f}")
print(f"  mask_diameter:            {MASK_SAMPLER.mask_diameter:.6f}")
print(f"  mask_radius:              {MASK_SAMPLER.mask_radius:.6f}")
print(f"  n_tok_per_channel:        {MASK_SAMPLER.n_tok_per_channel}")

check_mask(MASK_SAMPLER, "Cz")


In [ ]:
def expand_channel_mask_to_token_mask(mask_ch: torch.Tensor, n_tok_per_channel: int) -> torch.Tensor:
    """Expand a channel-level mask to token level.

    Supports either a single sample `(C,)` or a batch `(B, C)`.
    """
    if mask_ch.ndim == 1:
        return mask_ch.unsqueeze(-1).expand(-1, n_tok_per_channel).reshape(-1)
    if mask_ch.ndim == 2:
        return mask_ch.unsqueeze(-1).expand(-1, -1, n_tok_per_channel).reshape(mask_ch.shape[0], -1)
    raise ValueError(f"mask_ch must have ndim 1 or 2, got {mask_ch.ndim}")


def gather_visible_tokens_single(tokens: torch.Tensor, mask_tok: torch.Tensor) -> torch.Tensor:
    """Gather visible tokens for one sample.

    Parameters
    ----------
    tokens : FloatTensor `(N, D)`
    mask_tok : BoolTensor `(N,)`
    """
    return tokens[~mask_tok]


def gather_masked_tokens_single(tokens: torch.Tensor, mask_tok: torch.Tensor) -> torch.Tensor:
    """Gather masked tokens for one sample.

    Parameters
    ----------
    tokens : FloatTensor `(N, D)`
    mask_tok : BoolTensor `(N,)`
    """
    return tokens[mask_tok]

In [ ]:
_, _fake_mask_tok, _fake_mask_info = MASK_SAMPLER.sample(2, device="cpu")

_fake_tokens = torch.randn(TOTAL_TOKENS, EMB_DIM)
_fake_visible = gather_visible_tokens_single(_fake_tokens, _fake_mask_tok[0])
_fake_masked = gather_masked_tokens_single(_fake_tokens, _fake_mask_tok[0])

print("Token-routing:")
print(f"  Full token shape:         {tuple(_fake_tokens.shape)}")
print(f"  Visible token shape:      {tuple(_fake_visible.shape)}")
print(f"  Masked token shape:       {tuple(_fake_masked.shape)}")
print(f"  Sample masked channels:   {_fake_mask_info['masked_channel_counts'][0]}")
print("Single-sample token routing validated OK.")

In [ ]:
class MaskedTokenPredictor(nn.Module):
    """Predicts masked token representations from student context.

    Architecture: Transformer decoder (cross-attention).

    Queries  = masked positional encodings  (B, N_masked, D)
    Keys/Vals = student contextual features (B, N_vis, D)
    Output   = predicted masked embeddings  (B, N_masked, D)
    """

    def __init__(
        self,
        d_model: int,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.0,
    ):
        super().__init__()
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, student_context: torch.Tensor, masked_pos_enc: torch.Tensor) -> torch.Tensor:
        """Predict masked token embeddings."""
        decoded = self.decoder(tgt=masked_pos_enc, memory=student_context)
        return self.output_norm(decoded)

In [ ]:
PREDICTOR = MaskedTokenPredictor(
    d_model=EMB_DIM,
    nhead=CONFIG["predictor_nhead"],
    num_layers=CONFIG["predictor_n_layers"],
    dim_feedforward=CONFIG["predictor_dim_feedforward"],
).to(DEVICE)

predictor_param_count = sum(parameter.numel() for parameter in PREDICTOR.parameters())
print("MaskedTokenPredictor initialized:")
print(f"  d_model:          {EMB_DIM}")
print(f"  nhead:            {CONFIG['predictor_nhead']}")
print(f"  num_layers:       {CONFIG['predictor_n_layers']}")
print(f"  dim_feedforward:  {CONFIG['predictor_dim_feedforward']}")
print(f"  Total parameters: {predictor_param_count:,}")

_, _test_mask_tok, _ = MASK_SAMPLER.sample(1, device=DEVICE)
_test_n_masked = int(_test_mask_tok[0].sum().item())
_test_n_visible = int((~_test_mask_tok[0]).sum().item())

_fake_student_context = torch.randn(1, _test_n_visible, EMB_DIM, device=DEVICE)
_fake_masked_pos = torch.randn(1, _test_n_masked, EMB_DIM, device=DEVICE)

PREDICTOR.eval()
with torch.no_grad():
    _pred_out = PREDICTOR(_fake_student_context, _fake_masked_pos)

if _pred_out.shape != (1, _test_n_masked, EMB_DIM):
    raise RuntimeError(
        f"Predictor output shape mismatch: got {tuple(_pred_out.shape)} expected {(1, _test_n_masked, EMB_DIM)}"
    )

print("\nPredictor shape test: OK")
print(f"  student_context: {tuple(_fake_student_context.shape)}")
print(f"  masked_pos_enc:  {tuple(_fake_masked_pos.shape)}")
print(f"  prediction:      {tuple(_pred_out.shape)}")
PREDICTOR.train()

# 6. Training

In [ ]:
def masked_l1_loss(predicted: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """L1 loss on masked-token embeddings only."""
    return F.l1_loss(predicted, target)


@torch.no_grad()
def ema_update(student: nn.Module, teacher: nn.Module, ema_decay: float) -> None:
    """Update teacher parameters by exponential moving average from the student."""
    for student_param, teacher_param in zip(student.parameters(), teacher.parameters()):
        teacher_param.data.mul_(ema_decay).add_(student_param.data, alpha=1.0 - ema_decay)


def compute_single_sample_loss(x_single, student, teacher, predictor, mask_sampler, device, debug=False):
    """Compute the masked-token prediction loss for one sample."""
    x_single = x_single.unsqueeze(0).to(device)
    _, mask_tok, mask_info = mask_sampler.sample(1, device=device)
    mask_tok_single = mask_tok[0]

    if not bool(mask_tok_single.any()):
        raise RuntimeError("Mask sampler produced zero masked tokens.")
    if not bool((~mask_tok_single).any()):
        raise RuntimeError("Mask sampler masked every token in the sample.")

    teacher.eval()

    # Teacher target path uses current student local tokens + EMA contextual encoder.
    with torch.no_grad():
        teacher_local = student.feature_encoder(x_single)
        teacher_pos = student.pos_encoder(teacher_local)
        teacher_full_tokens = teacher_local + teacher_pos
        teacher_context = teacher.transformer.encoder(teacher_full_tokens)
        teacher_masked = teacher_context[:, mask_tok_single, :]

    student_local = student.feature_encoder(x_single)
    student_pos = student.pos_encoder(student_local)
    student_full_tokens = student_local + student_pos
    student_visible_tokens = student_full_tokens[:, ~mask_tok_single, :]
    masked_positional_queries = student_pos[:, mask_tok_single, :]
    student_context = student.transformer.encoder(student_visible_tokens)
    predicted_masked = predictor(student_context, masked_positional_queries)

    # Fail fast on any non-finite intermediate before loss/backprop.
    assert_finite_tensor("teacher_local", teacher_local)
    assert_finite_tensor("teacher_pos", teacher_pos)
    assert_finite_tensor("teacher_full_tokens", teacher_full_tokens)
    assert_finite_tensor("teacher_context", teacher_context)
    assert_finite_tensor("teacher_masked", teacher_masked)
    assert_finite_tensor("student_local", student_local)
    assert_finite_tensor("student_pos", student_pos)
    assert_finite_tensor("student_full_tokens", student_full_tokens)
    assert_finite_tensor("student_visible_tokens", student_visible_tokens)
    assert_finite_tensor("student_context", student_context)
    assert_finite_tensor("masked_positional_queries", masked_positional_queries)
    assert_finite_tensor("predicted_masked", predicted_masked)

    loss = masked_l1_loss(predicted_masked, teacher_masked.detach())
    assert_finite_tensor("loss", loss)

    sample_stats = {
        "center_channel": int(mask_info["center_indices"][0]),
        "masked_channels": int(mask_info["masked_channel_counts"][0]),
        "masked_tokens": int(mask_info["masked_token_counts"][0]),
        "visible_tokens": int((~mask_tok_single).sum().item()),
    }

    if debug:
        print("  [debug] first-sample training path:")
        print(f"    x_single:             {tuple(x_single.shape)}")
        print(f"    teacher_full_tokens:  {tuple(teacher_full_tokens.shape)}")
        print(f"    student_visible:      {tuple(student_visible_tokens.shape)}")
        print(f"    masked_queries:       {tuple(masked_positional_queries.shape)}")
        print(f"    predicted_masked:     {tuple(predicted_masked.shape)}")
        print(f"    teacher_masked:       {tuple(teacher_masked.shape)}")
        print(f"    center_channel:       {sample_stats['center_channel']}")
        print(f"    masked_channels:      {sample_stats['masked_channels']}")
        print(f"    masked_tokens:        {sample_stats['masked_tokens']}")
        print(f"    visible_tokens:       {sample_stats['visible_tokens']}")
        print(f"    sample_loss:          {loss.item():.6f}")

    return loss, sample_stats


def _aggregate_batch_stats(sample_stats):
    return {
        "mean_masked_channels": float(sum(s["masked_channels"] for s in sample_stats) / len(sample_stats)),
        "mean_masked_tokens": float(sum(s["masked_tokens"] for s in sample_stats) / len(sample_stats)),
        "mean_visible_tokens": float(sum(s["visible_tokens"] for s in sample_stats) / len(sample_stats)),
        "sample_stats": sample_stats,
    }


def process_batch_low_memory(
    X_cpu,
    student,
    teacher,
    predictor,
    mask_sampler,
    optimizer,
    ema_decay,
    device,
    training,
    debug=False,
    apply_update=True,
    backprop=True,
):
    """Process one batch with minimal peak memory by handling one sample at a time."""
    n_samples = int(X_cpu.shape[0])
    if n_samples == 0:
        raise RuntimeError("Received an empty batch.")

    sample_stats = []
    running_loss = 0.0

    if training and apply_update:
        optimizer.zero_grad(set_to_none=True)

    for sample_idx in range(n_samples):
        x_sample = X_cpu[sample_idx].float()
        sample_loss, stats = compute_single_sample_loss(
            x_sample,
            student=student,
            teacher=teacher,
            predictor=predictor,
            mask_sampler=mask_sampler,
            device=device,
            debug=debug and sample_idx == 0,
        )

        if not torch.isfinite(sample_loss):
            raise RuntimeError("Non-finite sample loss detected.")

        if training and apply_update and backprop:
            (sample_loss / n_samples).backward()

        running_loss += float(sample_loss.detach().item())
        sample_stats.append(stats)

        del sample_loss, x_sample

    if training and apply_update:
        optimizer.step()
        ema_update(student.transformer.encoder, teacher.transformer.encoder, ema_decay)

    batch_stats = _aggregate_batch_stats(sample_stats)
    return {
        "loss": float(running_loss / n_samples),
        **batch_stats,
    }


def pretraining_step(
    X,
    student,
    teacher,
    predictor,
    mask_sampler,
    optimizer,
    ema_decay,
    device,
    debug=False,
):
    student.train()
    predictor.train()
    teacher.eval()

    batch_result = process_batch_low_memory(
        X_cpu=X,
        student=student,
        teacher=teacher,
        predictor=predictor,
        mask_sampler=mask_sampler,
        optimizer=optimizer,
        ema_decay=ema_decay,
        device=device,
        training=True,
        debug=debug,
        apply_update=True,
        backprop=True,
    )

    return {
        "loss": batch_result["loss"],
        "mean_masked_channels": batch_result["mean_masked_channels"],
        "mean_masked_tokens": batch_result["mean_masked_tokens"],
        "mean_visible_tokens": batch_result["mean_visible_tokens"],
    }


def evaluate_pretraining_batch(X, student, teacher, predictor, mask_sampler, device):
    """Evaluate one validation batch without parameter updates."""
    student.eval()
    predictor.eval()
    teacher.eval()

    with torch.no_grad():
        batch_result = process_batch_low_memory(
            X_cpu=X,
            student=student,
            teacher=teacher,
            predictor=predictor,
            mask_sampler=mask_sampler,
            optimizer=None,
            ema_decay=None,
            device=device,
            training=False,
            debug=False,
            apply_update=False,
            backprop=False,
        )

    return batch_result

In [ ]:
OPTIMIZER = torch.optim.Adam(
    list(STUDENT.parameters()) + list(PREDICTOR.parameters()),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

_debug_train_subject_chunk = next(
    iter_subject_chunks(
        TRAIN_SUBJECTS,
        chunk_size=CONFIG["train_subject_chunk_size"],
        shuffle=False,
    )
)
_debug_train_windows, _debug_train_raw, _ = prepare_pretraining_chunk_dataset(
    paradigm_names=CONFIG["paradigm_names"],
    subject_ids=_debug_train_subject_chunk,
    split_name="debug-train",
    preprocessors=_PREPROCESSORS,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
    preload=CONFIG["window_preload"],
)
_debug_train_loader = make_chunk_dataloader(
    windows_dataset=_debug_train_windows,
    batch_size=CONFIG["batch_size"],
    training=True,
)

_debug_val_subject_chunk = next(
    iter_subject_chunks(
        VAL_SUBJECTS,
        chunk_size=CONFIG["val_subject_chunk_size"],
        shuffle=False,
    )
)
_debug_val_windows, _debug_val_raw, _ = prepare_pretraining_chunk_dataset(
    paradigm_names=CONFIG["paradigm_names"],
    subject_ids=_debug_val_subject_chunk,
    split_name="debug-validation",
    preprocessors=_PREPROCESSORS,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
    preload=CONFIG["window_preload"],
)
_debug_val_loader = make_chunk_dataloader(
    windows_dataset=_debug_val_windows,
    batch_size=CONFIG["batch_size"],
    training=False,
)

print("Optimizer: Adam")
print(f"Learning rate:             {CONFIG['learning_rate']}")
print(f"Weight decay:              {CONFIG['weight_decay']}")
print(f"Debug train chunk subjects: {_debug_train_subject_chunk}")
print(f"Debug train windows:        {len(_debug_train_windows)}")
print(f"Debug val chunk subjects:   {_debug_val_subject_chunk}")
print(f"Debug val windows:          {len(_debug_val_windows)}")

print("\nRunning first-batch debug (dry-run before optimizer step)...")
set_seed(CONFIG["seed"])

_debug_batch = next(iter(_debug_train_loader))
_debug_X = _debug_batch[0].float()

# Dry-run finite check: one-sample-at-a-time forward path without parameter updates.
_dry_run_result = process_batch_low_memory(
    X_cpu=_debug_X,
    student=STUDENT,
    teacher=TEACHER,
    predictor=PREDICTOR,
    mask_sampler=MASK_SAMPLER,
    optimizer=OPTIMIZER,
    ema_decay=CONFIG["ema_decay"],
    device=DEVICE,
    training=True,
    debug=True,
    apply_update=False,
    backprop=False,
)
if not np.isfinite(_dry_run_result["loss"]):
    raise RuntimeError("Dry-run first training batch produced a non-finite loss.")
print(f"  Dry-run first training-batch loss: {_dry_run_result['loss']:.6f}")

# Do not run a real optimizer update in the debug cell.
# The first parameter update should occur inside the real epoch loop.
_debug_train_result = _dry_run_result
print(f"  First training-batch loss (dry-run only): {_debug_train_result['loss']:.6f}")
print(f"  Mean masked channels:                    {_debug_train_result['mean_masked_channels']:.2f}")
print(f"  Mean masked tokens:                      {_debug_train_result['mean_masked_tokens']:.2f}")

_validation_batch = next(iter(_debug_val_loader))
_validation_result = evaluate_pretraining_batch(
    X=_validation_batch[0].float(),
    student=STUDENT,
    teacher=TEACHER,
    predictor=PREDICTOR,
    mask_sampler=MASK_SAMPLER,
    device=DEVICE,
)
print(f"  First validation-batch loss: {_validation_result['loss']:.6f}")

if not np.isfinite(_debug_train_result["loss"]):
    raise RuntimeError("First training batch produced a non-finite loss.")
if not np.isfinite(_validation_result["loss"]):
    raise RuntimeError("First validation batch produced a non-finite loss.")

del _debug_train_loader, _debug_val_loader
del _debug_train_windows, _debug_train_raw, _debug_val_windows, _debug_val_raw
gc.collect()

In [ ]:
import time


def build_backbone_export_metadata():
    """Metadata needed to reconstruct the pretrained student backbone later."""
    return {
        "conv_layers_spec": [list(spec) for spec in _DEFAULT_CONV_LAYER_SPEC],
        "sfreq": CONFIG["sfreq"],
        "input_window_seconds": INPUT_WINDOW_SECONDS,
        "chs_info": CHS_INFO,
        "ch_names": CH_NAMES,
        "token_geometry": {
            "n_channels": N_CHANNELS,
            "n_tok_per_channel": N_TOK_PER_CHANNEL,
            "emb_dim": EMB_DIM,
            "total_tokens": TOTAL_TOKENS,
        },
        "masking_config": {
            "strategy": CONFIG["masking_strategy"],
            "mask_diameter_percent": CONFIG["mask_diameter_percent"],
            "head_diameter": MASK_SAMPLER.head_diameter,
            "mask_diameter": MASK_SAMPLER.mask_diameter,
            "mask_radius": MASK_SAMPLER.mask_radius,
        },
        "preprocessing_config": {
            "sfreq": CONFIG["sfreq"],
            "filter_low": CONFIG["filter_low"],
            "filter_high": CONFIG["filter_high"],
            "pretrain_window_samples": CONFIG.get("pretrain_window_samples"),
            "pretrain_duration_s": CONFIG["pretrain_duration_s"],
            "sampling_interval_s": CONFIG["sampling_interval_s"],
            "window_size_samples": WINDOW_SIZE_SAMPLES,
            "sampling_interval_samples": SAMPLING_INTERVAL_SAMPLES,
            "pretrain_window_mode": CONFIG.get("pretrain_window_mode"),
            "pretrain_start_s": CONFIG.get("pretrain_start_s"),
            "pretrain_stop_s": CONFIG.get("pretrain_stop_s"),
        },
        "subject_split": {
            "train_subject_ids": TRAIN_SUBJECTS,
            "val_subject_ids": VAL_SUBJECTS,
            "excluded_subject_ids": DOWNSTREAM_ONLY_SUBJECTS,
        },
        "export_config": {
            "export_with_chans": bool(CONFIG.get("export_with_chans", True)),
            "export_without_chans": bool(CONFIG.get("export_without_chans", True)),
        },
    }


BACKBONE_EXPORT_METADATA = build_backbone_export_metadata()


def strip_channel_specific_state_dict(state_dict):
    """Remove channel-specific/position-specific weights for a no-channel export.

    This preserves the trained backbone weights but drops keys that are likely tied
    to a fixed channel set. The full checkpoint is also saved, so this stripped
    export is only for downstream setups that need fresh channel embeddings.
    """
    stripped = {}
    removed = []
    for key, value in state_dict.items():
        key_lower = key.lower()
        is_channel_specific = (
            "channel" in key_lower
            or "chan" in key_lower
            or "ch_" in key_lower
            or "chs" in key_lower
            or "pos_encoder" in key_lower
        )
        if is_channel_specific:
            removed.append(key)
        else:
            stripped[key] = value
    return stripped, removed


def save_checkpoint(artifact_dir, tag, student, teacher, predictor, optimizer, epoch, epoch_record, metrics_so_far):
    """Save a full training checkpoint."""
    checkpoint = {
        "epoch": epoch,
        "epoch_record": epoch_record,
        "student_state_dict": student.state_dict(),
        "teacher_state_dict": teacher.state_dict(),
        "predictor_state_dict": predictor.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics_so_far,
        "backbone_export_metadata": BACKBONE_EXPORT_METADATA,
    }
    checkpoint_path = Path(artifact_dir) / f"checkpoint_{tag}.pt"
    torch.save(checkpoint, checkpoint_path)
    return str(checkpoint_path)


def save_student_backbone_export(artifact_dir, tag, student):
    """Save full and no-channel student-backbone exports for downstream use."""
    export_paths = {}

    if bool(CONFIG.get("export_with_chans", True)):
        export_payload = {
            "student_backbone_state_dict": student.state_dict(),
            "export_variant": "with_chans",
            **BACKBONE_EXPORT_METADATA,
        }
        export_path = Path(artifact_dir) / f"student_backbone_with_chans_{tag}.pt"
        torch.save(export_payload, export_path)
        export_paths["with_chans"] = str(export_path)

        # Backward-compatible filename expected by earlier notebooks.
        legacy_path = Path(artifact_dir) / f"student_backbone_{tag}.pt"
        torch.save(export_payload, legacy_path)
        export_paths["legacy"] = str(legacy_path)

    if bool(CONFIG.get("export_without_chans", True)):
        stripped_state, removed_keys = strip_channel_specific_state_dict(student.state_dict())
        export_payload = {
            "student_backbone_state_dict": stripped_state,
            "removed_channel_specific_keys": removed_keys,
            "export_variant": "without_chans",
            **BACKBONE_EXPORT_METADATA,
        }
        export_path = Path(artifact_dir) / f"student_backbone_without_chans_{tag}.pt"
        torch.save(export_payload, export_path)
        export_paths["without_chans"] = str(export_path)

    return export_paths


def append_epoch_metrics(artifact_dir, epoch_record):
    """Append one epoch record to epoch_metrics.jsonl."""
    epoch_metrics_path = Path(artifact_dir) / "epoch_metrics.jsonl"
    with open(epoch_metrics_path, "a") as metric_file:
        metric_file.write(json.dumps(epoch_record) + "\n")


In [ ]:
def run_pretraining_epoch_chunked(
    epoch,
    split_name,
    subject_ids,
    chunk_size,
    chunk_shuffle,
    paradigm_names,
    preprocessors,
    window_size_samples,
    sampling_interval_samples,
    preload,
    batch_size,
    training,
    student,
    teacher,
    predictor,
    mask_sampler,
    optimizer,
    ema_decay,
    device,
    seed,
):
    """Run one train or validation epoch by streaming subject chunks."""
    epoch_start = time.time()
    batch_losses = []
    masked_channels = []
    masked_tokens = []
    visible_tokens = []

    chunk_losses = []
    chunk_window_counts = []
    chunk_recording_counts = []
    n_batches_total = 0

    chunk_subject_lists = list(
        iter_subject_chunks(
            subject_ids=subject_ids,
            chunk_size=chunk_size,
            shuffle=chunk_shuffle and training,
            seed=seed + int(epoch),
        )
    )
    if len(chunk_subject_lists) == 0:
        raise RuntimeError(f"{split_name} subject split produced zero chunks.")

    log_debug(
        f"[debug] [{split_name}] epoch {epoch}: "
        f"chunks={len(chunk_subject_lists)} chunk_size={chunk_size} "
        f"shuffle={bool(chunk_shuffle and training)} preload={preload}"
    )

    for chunk_idx, chunk_subjects in enumerate(chunk_subject_lists, start=1):
        chunk_windows = None
        chunk_raw_dataset = None
        chunk_loader = None
        chunk_start_time = time.time()
        try:
            log_debug(
                f"[debug] [{split_name}] epoch {epoch} chunk {chunk_idx}/{len(chunk_subject_lists)} "
                f"subjects={chunk_subjects} paradigms={list(paradigm_names)}"
            )

            chunk_windows, chunk_raw_dataset, chunk_recordings_by_paradigm = prepare_pretraining_chunk_dataset(
                paradigm_names=paradigm_names,
                subject_ids=chunk_subjects,
                split_name=split_name,
                preprocessors=preprocessors,
                window_size_samples=window_size_samples,
                sampling_interval_samples=sampling_interval_samples,
                preload=preload,
            )

            chunk_recording_count = len(chunk_raw_dataset.datasets)
            chunk_window_count = len(chunk_windows)
            chunk_loader = make_chunk_dataloader(
                windows_dataset=chunk_windows,
                batch_size=batch_size,
                training=training,
            )

            per_chunk_batch_losses = []
            per_chunk_masked_channels = []
            per_chunk_masked_tokens = []
            per_chunk_visible_tokens = []

            for batch in chunk_loader:
                X = batch[0].float()
                if training:
                    batch_result = pretraining_step(
                        X=X,
                        student=student,
                        teacher=teacher,
                        predictor=predictor,
                        mask_sampler=mask_sampler,
                        optimizer=optimizer,
                        ema_decay=ema_decay,
                        device=device,
                        debug=False,
                    )
                else:
                    batch_result = evaluate_pretraining_batch(
                        X=X,
                        student=student,
                        teacher=teacher,
                        predictor=predictor,
                        mask_sampler=mask_sampler,
                        device=device,
                    )

                loss_value = float(batch_result["loss"])
                per_chunk_batch_losses.append(loss_value)
                per_chunk_masked_channels.append(float(batch_result["mean_masked_channels"]))
                per_chunk_masked_tokens.append(float(batch_result["mean_masked_tokens"]))
                per_chunk_visible_tokens.append(float(batch_result["mean_visible_tokens"]))

                batch_losses.append(loss_value)
                masked_channels.append(float(batch_result["mean_masked_channels"]))
                masked_tokens.append(float(batch_result["mean_masked_tokens"]))
                visible_tokens.append(float(batch_result["mean_visible_tokens"]))
                n_batches_total += 1

            if len(per_chunk_batch_losses) == 0:
                raise RuntimeError(
                    f"{split_name} chunk {chunk_subjects} produced zero batches; "
                    "adjust chunk size or batch_size/drop_last settings."
                )

            chunk_mean_loss = float(np.mean(per_chunk_batch_losses))
            chunk_losses.append(chunk_mean_loss)
            chunk_window_counts.append(chunk_window_count)
            chunk_recording_counts.append(chunk_recording_count)

            chunk_elapsed = time.time() - chunk_start_time
            print(
                f"[{split_name}] epoch {epoch} chunk {chunk_idx}/{len(chunk_subject_lists)} "
                f"subjects={chunk_subjects} windows={chunk_window_count} "
                f"batches={len(per_chunk_batch_losses)} loss={chunk_mean_loss:.6f} "
                f"running={float(np.mean(batch_losses)):.6f} "
                f"time={chunk_elapsed:.1f}s"
            )

        finally:
            del chunk_loader, chunk_windows, chunk_raw_dataset
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not batch_losses:
        raise RuntimeError(f"{split_name} epoch produced zero batches across all chunks.")

    epoch_time_s = time.time() - epoch_start
    return {
        "epoch": int(epoch),
        "split": split_name,
        "mean_loss": float(np.mean(batch_losses)),
        "min_loss": float(np.min(batch_losses)),
        "max_loss": float(np.max(batch_losses)),
        "mean_masked_channels": float(np.mean(masked_channels)),
        "mean_masked_tokens": float(np.mean(masked_tokens)),
        "mean_visible_tokens": float(np.mean(visible_tokens)),
        "n_batches": int(len(batch_losses)),
        "n_chunks": int(len(chunk_subject_lists)),
        "mean_chunk_loss": float(np.mean(chunk_losses)),
        "mean_chunk_windows": float(np.mean(chunk_window_counts)),
        "mean_chunk_recordings": float(np.mean(chunk_recording_counts)),
        "epoch_time_s": round(epoch_time_s, 2),
    }

In [ ]:
_train_chunk_count = len(list(iter_subject_chunks(
    TRAIN_SUBJECTS,
    chunk_size=CONFIG["train_subject_chunk_size"],
    shuffle=False,
)))
_val_chunk_count = len(list(iter_subject_chunks(
    VAL_SUBJECTS,
    chunk_size=CONFIG["val_subject_chunk_size"],
    shuffle=False,
)))

print("=" * 70)
print("[setup] STARTING LIU2024 S-JEPA PRETRAINING")
print("=" * 70)
print(f"[setup] run_id:                  {RUN_ID}")
print(f"[setup] seed:                    {CONFIG['seed']}")
print(f"[setup] device:                  {DEVICE}")
print(f"[setup] paradigms:               {CONFIG['paradigm_names']}")
print(f"[setup] train_subjects:          {len(TRAIN_SUBJECTS)}  {TRAIN_SUBJECTS}")
print(f"[setup] val_subjects:            {len(VAL_SUBJECTS)}  {VAL_SUBJECTS}")
print(f"[setup] excluded_subjects:       {DOWNSTREAM_ONLY_SUBJECTS}")
print(f"[setup] train_chunk_size:        {CONFIG['train_subject_chunk_size']}")
print(f"[setup] val_chunk_size:          {CONFIG['val_subject_chunk_size']}")
print(f"[setup] train_chunks_per_epoch:  {_train_chunk_count}")
print(f"[setup] val_chunks_per_epoch:    {_val_chunk_count}")
print(f"[setup] batch_size:              {CONFIG['batch_size']}")
print(f"[setup] early_stopping_patience: {CONFIG['early_stopping_patience']}")
print(f"[setup] window_preload:          {CONFIG['window_preload']}")
print(f"[setup] pretrain_duration:       {CONFIG['pretrain_duration_s']} s")
print(f"[setup] sampling_interval:       {CONFIG['sampling_interval_s']} s")
print(f"[setup] mask_diameter:           {CONFIG['mask_diameter_percent']}%")
print(f"[setup] ema_decay:               {CONFIG['ema_decay']}")
print(f"[setup] epoch_limit:             {CONFIG['n_epochs']}")
print(f"[setup] log_verbosity:           {LOG_VERBOSITY}")
print("=" * 70)

set_seed(CONFIG["seed"])

EPOCH_METRICS = []
BEST_VAL_LOSS = float("inf")
BEST_EPOCH = -1
PATIENCE_COUNTER = 0
STOP_REASON = "max_epochs_reached"

for epoch in range(1, CONFIG["n_epochs"] + 1):
    train_metrics = run_pretraining_epoch_chunked(
        epoch=epoch,
        split_name="train",
        subject_ids=TRAIN_SUBJECTS,
        chunk_size=CONFIG["train_subject_chunk_size"],
        chunk_shuffle=CONFIG["chunk_shuffle"],
        paradigm_names=CONFIG["paradigm_names"],
        preprocessors=_PREPROCESSORS,
        window_size_samples=WINDOW_SIZE_SAMPLES,
        sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
        preload=CONFIG["window_preload"],
        batch_size=CONFIG["batch_size"],
        training=True,
        student=STUDENT,
        teacher=TEACHER,
        predictor=PREDICTOR,
        mask_sampler=MASK_SAMPLER,
        optimizer=OPTIMIZER,
        ema_decay=CONFIG["ema_decay"],
        device=DEVICE,
        seed=CONFIG["seed"],
    )
    val_metrics = run_pretraining_epoch_chunked(
        epoch=epoch,
        split_name="val",
        subject_ids=VAL_SUBJECTS,
        chunk_size=CONFIG["val_subject_chunk_size"],
        chunk_shuffle=False,
        paradigm_names=CONFIG["paradigm_names"],
        preprocessors=_PREPROCESSORS,
        window_size_samples=WINDOW_SIZE_SAMPLES,
        sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
        preload=CONFIG["window_preload"],
        batch_size=CONFIG["batch_size"],
        training=False,
        student=STUDENT,
        teacher=TEACHER,
        predictor=PREDICTOR,
        mask_sampler=MASK_SAMPLER,
        optimizer=OPTIMIZER,
        ema_decay=CONFIG["ema_decay"],
        device=DEVICE,
        seed=CONFIG["seed"],
    )

    improved = val_metrics["mean_loss"] < BEST_VAL_LOSS
    if improved:
        BEST_VAL_LOSS = val_metrics["mean_loss"]
        BEST_EPOCH = epoch
        PATIENCE_COUNTER = 0
    else:
        PATIENCE_COUNTER += 1

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_metrics["mean_loss"],
        "val_loss": val_metrics["mean_loss"],
        "train_masked_channels": train_metrics["mean_masked_channels"],
        "val_masked_channels": val_metrics["mean_masked_channels"],
        "train_masked_tokens": train_metrics["mean_masked_tokens"],
        "val_masked_tokens": val_metrics["mean_masked_tokens"],
        "train_visible_tokens": train_metrics["mean_visible_tokens"],
        "val_visible_tokens": val_metrics["mean_visible_tokens"],
        "train_batches": train_metrics["n_batches"],
        "val_batches": val_metrics["n_batches"],
        "train_chunks": train_metrics["n_chunks"],
        "val_chunks": val_metrics["n_chunks"],
        "train_time_s": train_metrics["epoch_time_s"],
        "val_time_s": val_metrics["epoch_time_s"],
        "best_val_loss": BEST_VAL_LOSS,
        "best_epoch": BEST_EPOCH,
        "patience_counter": PATIENCE_COUNTER,
        "improved_val": improved,
    }
    EPOCH_METRICS.append(epoch_record)
    append_epoch_metrics(ARTIFACT_DIR, epoch_record)

    save_checkpoint(
        ARTIFACT_DIR,
        "latest",
        STUDENT,
        TEACHER,
        PREDICTOR,
        OPTIMIZER,
        epoch,
        epoch_record,
        EPOCH_METRICS,
    )
    save_student_backbone_export(ARTIFACT_DIR, "latest", STUDENT)

    if improved:
        save_checkpoint(
            ARTIFACT_DIR,
            "best",
            STUDENT,
            TEACHER,
            PREDICTOR,
            OPTIMIZER,
            epoch,
            epoch_record,
            EPOCH_METRICS,
        )
        save_student_backbone_export(ARTIFACT_DIR, "best", STUDENT)

    print(
        f"[epoch] {epoch:3d}/{CONFIG['n_epochs']}  "
        f"train_loss={train_metrics['mean_loss']:.6f}  "
        f"val_loss={val_metrics['mean_loss']:.6f}  "
        f"best_val={BEST_VAL_LOSS:.6f}  "
        f"patience={PATIENCE_COUNTER}/{CONFIG['early_stopping_patience']}  "
        f"train_chunks={train_metrics['n_chunks']}  "
        f"val_chunks={val_metrics['n_chunks']}  "
        f"train_time={train_metrics['epoch_time_s']:.1f}s  "
        f"val_time={val_metrics['epoch_time_s']:.1f}s"
    )

    if PATIENCE_COUNTER >= CONFIG["early_stopping_patience"]:
        STOP_REASON = "early_stopping"
        print(
            f"[epoch] early stopping triggered after epoch {epoch}: "
            f"no validation improvement for {CONFIG['early_stopping_patience']} epochs."
        )
        break

print("=" * 70)
print(f"[epoch] training complete. Best val loss: {BEST_VAL_LOSS:.6f} at epoch {BEST_EPOCH}.")
print(f"[epoch] stop reason: {STOP_REASON}")
print("=" * 70)

## 6.1. Loss Curve Summary

In [ ]:
print("=" * 70)
print("LOSS HISTORY SUMMARY")
print("=" * 70)

if EPOCH_METRICS:
    train_losses = [record["train_loss"] for record in EPOCH_METRICS]
    val_losses = [record["val_loss"] for record in EPOCH_METRICS]
    total_train_time = sum(record["train_time_s"] for record in EPOCH_METRICS)
    total_val_time = sum(record["val_time_s"] for record in EPOCH_METRICS)

    print(f"  Epochs completed:     {len(EPOCH_METRICS)}")
    print(f"  First train loss:     {train_losses[0]:.6f}")
    print(f"  Last train loss:      {train_losses[-1]:.6f}")
    print(f"  First val loss:       {val_losses[0]:.6f}")
    print(f"  Last val loss:        {val_losses[-1]:.6f}")
    print(f"  Best val loss:        {BEST_VAL_LOSS:.6f} (epoch {BEST_EPOCH})")
    print(f"  Total train time:     {total_train_time:.1f} s")
    print(f"  Total validation time:{total_val_time:.1f} s")
    print(f"  Stop reason:          {STOP_REASON}")
else:
    print("  No epoch metrics recorded.")

# 7. Diagnostics and Outputs

In [ ]:
def summarize_mask_coverage(mask_sampler: RandomSpatialBlockMaskSampler, n_samples: int = 200):
    """Sample mask coverage statistics for the current radius-based sampler."""
    masked_channel_counts = []
    masked_token_counts = []

    for _ in range(n_samples):
        _, _, mask_info = mask_sampler.sample(1, device="cpu")
        masked_channel_counts.append(mask_info["masked_channel_counts"][0])
        masked_token_counts.append(mask_info["masked_token_counts"][0])

    mask_stats = {
        "n_channels": mask_sampler.n_channels,
        "n_tok_per_channel": mask_sampler.n_tok_per_channel,
        "head_diameter": mask_sampler.head_diameter,
        "mask_diameter_percent": mask_sampler.mask_diameter_percent,
        "mask_diameter": mask_sampler.mask_diameter,
        "mask_radius": mask_sampler.mask_radius,
        "masked_channels_mean": float(np.mean(masked_channel_counts)),
        "masked_channels_min": int(np.min(masked_channel_counts)),
        "masked_channels_max": int(np.max(masked_channel_counts)),
        "masked_tokens_mean": float(np.mean(masked_token_counts)),
        "masked_tokens_min": int(np.min(masked_token_counts)),
        "masked_tokens_max": int(np.max(masked_token_counts)),
        "n_samples": int(n_samples),
    }

    print("Mask coverage statistics:")
    for key, value in mask_stats.items():
        print(f"  {key}: {value}")

    mask_stats_path = ARTIFACT_DIR / "mask_stats.json"
    with open(mask_stats_path, "w") as output_file:
        json.dump(mask_stats, output_file, indent=2)
    print(f"\nMask stats saved to: {mask_stats_path}")

    return mask_stats


MASK_STATS = summarize_mask_coverage(MASK_SAMPLER)

In [ ]:
def inspect_one_batch_end_to_end(student, teacher, predictor, mask_sampler, windows_dataset, device):
    """Inspect one validation batch end to end using the first sample in the batch."""
    loader = DataLoader(windows_dataset, batch_size=4, shuffle=False)
    batch = next(iter(loader))
    x_batch = batch[0].float().to(device)
    x_single = x_batch[0].unsqueeze(0)

    student.eval()
    teacher.eval()
    predictor.eval()

    mask_ch, mask_tok, mask_info = mask_sampler.sample(1, device=device)
    mask_tok_single = mask_tok[0]

    with torch.no_grad():
        teacher_local = teacher.feature_encoder(x_single)
        teacher_pos = teacher.pos_encoder(teacher_local)
        teacher_full = teacher_local + teacher_pos
        teacher_context = teacher.transformer.encoder(teacher_full)
        teacher_masked = teacher_context[:, mask_tok_single, :]

        student_local = student.feature_encoder(x_single)
        student_pos = student.pos_encoder(student_local)
        student_full = student_local + student_pos
        student_visible = student_full[:, ~mask_tok_single, :]
        masked_queries = student_pos[:, mask_tok_single, :]
        student_context = student.transformer.encoder(student_visible)
        predicted_masked = predictor(student_context, masked_queries)
        loss_value = masked_l1_loss(predicted_masked, teacher_masked)

    print("=" * 70)
    print("ONE-BATCH END-TO-END INSPECTION")
    print("=" * 70)
    print(f"  x_single:            {tuple(x_single.shape)}")
    print(f"  teacher_full:        {tuple(teacher_full.shape)}")
    print(f"  student_visible:     {tuple(student_visible.shape)}")
    print(f"  masked_queries:      {tuple(masked_queries.shape)}")
    print(f"  predicted_masked:    {tuple(predicted_masked.shape)}")
    print(f"  teacher_masked:      {tuple(teacher_masked.shape)}")
    print(f"  center_channel:      {mask_info['center_indices'][0]}")
    print(f"  masked_channels:     {mask_info['masked_channel_counts'][0]}")
    print(f"  masked_tokens:       {mask_info['masked_token_counts'][0]}")
    print(f"  inspection_loss:     {loss_value.item():.6f}")
    print("=" * 70)


_inspect_val_chunk = next(
    iter_subject_chunks(
        VAL_SUBJECTS,
        chunk_size=CONFIG["val_subject_chunk_size"],
        shuffle=False,
    )
)
_inspect_val_windows, _inspect_val_raw, _ = prepare_pretraining_chunk_dataset(
    paradigm_names=CONFIG["paradigm_names"],
    subject_ids=_inspect_val_chunk,
    split_name="validation-inspector",
    preprocessors=_PREPROCESSORS,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    sampling_interval_samples=SAMPLING_INTERVAL_SAMPLES,
    preload=CONFIG["window_preload"],
)
inspect_one_batch_end_to_end(
    STUDENT,
    TEACHER,
    PREDICTOR,
    MASK_SAMPLER,
    _inspect_val_windows,
    DEVICE,
)

del _inspect_val_windows, _inspect_val_raw
gc.collect()

In [ ]:
metrics_path = ARTIFACT_DIR / "metrics.json"
with open(metrics_path, "w") as output_file:
    json.dump(EPOCH_METRICS, output_file, indent=2)
print(f"Epoch metrics saved to:           {metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "paradigms": CONFIG["paradigm_names"],
    "subject_split": {
        "train_subject_ids": TRAIN_SUBJECTS,
        "val_subject_ids": VAL_SUBJECTS,
        "excluded_subject_ids": DOWNSTREAM_ONLY_SUBJECTS,
    },
    "streaming": {
        "train_subject_chunk_size": CONFIG["train_subject_chunk_size"],
        "val_subject_chunk_size": CONFIG["val_subject_chunk_size"],
        "chunk_shuffle": CONFIG["chunk_shuffle"],
        "window_preload": CONFIG["window_preload"],
        "train_chunks_per_epoch": _train_chunk_count,
        "val_chunks_per_epoch": _val_chunk_count,
    },
    "reference_dataset": {
        "reference_subjects": _reference_subject_chunk,
        "reference_recordings": len(REFERENCE_RAW_DATASET.datasets),
        "reference_windows": len(REFERENCE_WINDOWS_DATASET),
    },
    "preprocessing": BACKBONE_EXPORT_METADATA["preprocessing_config"],
    "masking": BACKBONE_EXPORT_METADATA["masking_config"],
    "model": BACKBONE_EXPORT_METADATA["token_geometry"],
    "training": {
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "weight_decay": CONFIG["weight_decay"],
        "ema_decay": CONFIG["ema_decay"],
        "early_stopping_patience": CONFIG["early_stopping_patience"],
        "epochs_completed": len(EPOCH_METRICS),
        "best_epoch": BEST_EPOCH,
        "best_val_loss": BEST_VAL_LOSS,
        "stop_reason": STOP_REASON,
        "device": str(DEVICE),
    },
    "artifacts": {
        "config": str(ARTIFACT_DIR / "config.json"),
        "run_log": str(LOG_PATH),
        "metrics": str(metrics_path),
        "epoch_metrics_jsonl": str(ARTIFACT_DIR / "epoch_metrics.jsonl"),
        "checkpoint_latest": str(ARTIFACT_DIR / "checkpoint_latest.pt"),
        "checkpoint_best": str(ARTIFACT_DIR / "checkpoint_best.pt"),
        "student_backbone_latest": str(ARTIFACT_DIR / "student_backbone_latest.pt"),
        "student_backbone_best": str(ARTIFACT_DIR / "student_backbone_best.pt"),
        "mask_stats": str(ARTIFACT_DIR / "mask_stats.json"),
    },
}

metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(metadata_path, "w") as output_file:
    json.dump(run_metadata, output_file, indent=2, default=str)
print(f"Run metadata saved to:            {metadata_path}")

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:            {RUN_ID}")
print(f"Artifacts:         {ARTIFACT_DIR}")
print()
print("Subject split:")
print(f"  Train:           {TRAIN_SUBJECTS}")
print(f"  Validation:      {VAL_SUBJECTS}")
print(f"  Excluded:        {DOWNSTREAM_ONLY_SUBJECTS}")
print()
print("Data streaming:")
print(f"  Paradigms:       {CONFIG['paradigm_names']}")
print(f"  Train chunk size:{CONFIG['train_subject_chunk_size']}")
print(f"  Val chunk size:  {CONFIG['val_subject_chunk_size']}")
print(f"  Train chunks:    {_train_chunk_count}")
print(f"  Val chunks:      {_val_chunk_count}")
print(f"  Preload windows: {CONFIG['window_preload']}")
print(f"  Window seconds:  {CONFIG['pretrain_duration_s']}")
print(f"  Interval secs:   {CONFIG['sampling_interval_s']}")
print()
print("Masking:")
print(f"  Diameter %:      {CONFIG['mask_diameter_percent']}")
print(f"  Radius:          {MASK_SAMPLER.mask_radius:.6f}")
print()
print("Results:")
if EPOCH_METRICS:
    print(f"  Epochs run:      {len(EPOCH_METRICS)}")
    print(f"  Best epoch:      {BEST_EPOCH}")
    print(f"  Best val loss:   {BEST_VAL_LOSS:.6f}")
    print(f"  Final val loss:  {EPOCH_METRICS[-1]['val_loss']:.6f}")
    print(f"  Stop reason:     {STOP_REASON}")
else:
    print("  No training epochs recorded.")
print()
print("Artifact files:")
for artifact_name, artifact_path in run_metadata["artifacts"].items():
    print(f"  {artifact_name}: {artifact_path}")
print("=" * 70)